In [14]:
import gdown
import joblib
import pandas as pd

# Google Drive file IDs
model_id = "12faKEQW-cc9HhaoPRUuO7I8fmId2oQXM"
le_team_id = "1ugHiBYpUuPOB-yCsyORjKUf9K5M0KN1d"
le_venue_id = "1LpMtBk8jYTfCKeyPTN3k6b2eoqxTuRUA"

# Define local paths
model_path = "final_rf_model.pkl"
le_team_path = "le_team.pkl"
le_venue_path = "le_venue.pkl"

# Download files
gdown.download(f"https://drive.google.com/uc?id={model_id}", model_path, quiet=False)
gdown.download(f"https://drive.google.com/uc?id={le_team_id}", le_team_path, quiet=False)
gdown.download(f"https://drive.google.com/uc?id={le_venue_id}", le_venue_path, quiet=False)

# Load files
model = joblib.load(model_path)
le_team = joblib.load(le_team_path)
le_venue = joblib.load(le_venue_path)

print("✅ All files downloaded and loaded successfully!\n")


Downloading...
From (original): https://drive.google.com/uc?id=12faKEQW-cc9HhaoPRUuO7I8fmId2oQXM
From (redirected): https://drive.google.com/uc?id=12faKEQW-cc9HhaoPRUuO7I8fmId2oQXM&confirm=t&uuid=0c8d55f0-efd6-44eb-93f6-4fa2f776e2d2
To: /content/final_rf_model.pkl
100%|██████████| 366M/366M [00:05<00:00, 65.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ugHiBYpUuPOB-yCsyORjKUf9K5M0KN1d
To: /content/le_team.pkl
100%|██████████| 877/877 [00:00<00:00, 1.23MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LpMtBk8jYTfCKeyPTN3k6b2eoqxTuRUA
To: /content/le_venue.pkl
100%|██████████| 1.88k/1.88k [00:00<00:00, 5.40MB/s]


✅ All files downloaded and loaded successfully!



/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [21]:

# Function to get user input (with hardcoded example input)
def get_user_input():
    user_input = {
        'inning': 2,
        'cum_runs': 160,
        'cum_wickets': 5,
        'overs_completed': 18.0,
        'target': 190,
        'batting_team': 'Mumbai Indians',
        'bowling_team': 'Chennai Super Kings',
        'venue': 'Wankhede Stadium'
    }
    return user_input

# Get user input
data = get_user_input()

# Encode categorical features
try:
    data['batting_team'] = le_team.transform([data['batting_team']])[0]
    data['bowling_team'] = le_team.transform([data['bowling_team']])[0]
    data['venue'] = le_venue.transform([data['venue']])[0]
except ValueError as e:
    print(f"Error: {e}. Ensure the team/venue exists in training data.")
    exit()

# Compute additional features
current_run_rate = data['cum_runs'] / data['overs_completed'] if data['overs_completed'] > 0 else 0
remaining_overs = 20 - data['overs_completed']
required_run_rate = (data['target'] - data['cum_runs']) / remaining_overs if (data['inning'] == 2 and remaining_overs > 0) else 0

# Create DataFrame for model input
input_df = pd.DataFrame([data])
input_df['current_run_rate'] = current_run_rate
input_df['required_run_rate'] = required_run_rate

# Keep only necessary columns
# Select only the features that match the model
expected_features = ['inning', 'cum_runs', 'overs_completed', 'target',
                     'batting_team', 'bowling_team', 'venue']

input_df = input_df[expected_features]  # Drop unnecessary columns


# Predict using the model
prediction = model.predict(input_df)[0]
predicted_probabilities = model.predict_proba(input_df)[0]

# Determine winner
predicted_winner = "Batting Team Wins" if prediction == 1 else "Bowling Team Wins"

# Output results
print("\n🏆 Predicted Result:")
print(f"Winner: {predicted_winner}")
print(f"Win Probability: {predicted_probabilities[1] * 100:.2f}%")
print(f"Loss Probability: {predicted_probabilities[0] * 100:.2f}%")


🏆 Predicted Result:
Winner: Batting Team Wins
Win Probability: 53.53%
Loss Probability: 46.47%


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
